# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [1]:
from IPython.display import Markdown, display

display(Markdown("""
# 1. Two Paper Findings + My Methodology Questions

## Finding 1 — Content Performance Curve

The paper reports that content performance peaks around 61–90 days,
declines after 270 days, and that some 365+ day content performs better
when it has been recently refreshed.

### Methodology question

Where does the refresh distinction come from, and does the observational
comparison adequately account for differences between refreshed and
unrefreshed pages?

The result supports an observed association between freshness/refresh status
and performance, but the validation design does not by itself establish that
refreshing caused the improvement. Other factors such as content quality,
existing visibility, topic demand, or page history could also contribute.

**Safe interpretation:** observed association, not causal proof.


## Finding 2 — Engagement and Visibility Move Together

The paper reports that high scroll and high engagement are associated with
higher health scores, and that stronger visibility consistency appears
alongside stronger engagement.

### Methodology question

Are engagement measures being interpreted as explanatory signals or as
correlated outcomes?

Because the study is observational, stronger search visibility could itself
lead to more engagement. Therefore, the direction of the relationship is
not established by the comparison alone.

**Safe interpretation:** measured association, not evidence that increasing
engagement will necessarily cause better search visibility.
"""))


# 1. Two Paper Findings + My Methodology Questions

## Finding 1 — Content Performance Curve

The paper reports that content performance peaks around 61–90 days,
declines after 270 days, and that some 365+ day content performs better
when it has been recently refreshed.

### Methodology question

Where does the refresh distinction come from, and does the observational
comparison adequately account for differences between refreshed and
unrefreshed pages?

The result supports an observed association between freshness/refresh status
and performance, but the validation design does not by itself establish that
refreshing caused the improvement. Other factors such as content quality,
existing visibility, topic demand, or page history could also contribute.

**Safe interpretation:** observed association, not causal proof.


## Finding 2 — Engagement and Visibility Move Together

The paper reports that high scroll and high engagement are associated with
higher health scores, and that stronger visibility consistency appears
alongside stronger engagement.

### Methodology question

Are engagement measures being interpreted as explanatory signals or as
correlated outcomes?

Because the study is observational, stronger search visibility could itself
lead to more engagement. Therefore, the direction of the relationship is
not established by the comparison alone.

**Safe interpretation:** measured association, not evidence that increasing
engagement will necessarily cause better search visibility.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
import pandas as pd
import duckdb
from huggingface_hub import hf_hub_download
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from IPython.display import Markdown, display

display(Markdown("""
# 2. My Model Under an Honest Split

## Validation design

The model is evaluated using a time-aware split. Earlier observations are
used for training and later observations are held out for testing.

This better reflects the real decision setting because the model should use
past information to make decisions about later observations.

The current experiment still uses the Week-4 baseline score as its measured
target. Therefore, this validation measures agreement with the baseline rule,
not future business performance.
"""))

# Download the same warehouse sample used in W05
parquet_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance_sample.parquet"
)

# Load June 2026 data
con = duckdb.connect()

df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_sessions,
    ga4_engaged_sessions
FROM read_parquet('{parquet_file}')
WHERE gsc_data_available IS TRUE
  AND report_date BETWEEN DATE '2026-06-01' AND DATE '2026-06-30'
""").df()

# Remove duplicate content/client/date observations
df = df.drop_duplicates(
    subset=["report_date", "client_hash_id", "content_hash_id"]
).copy()

# Convert date
df["report_date"] = pd.to_datetime(df["report_date"])

# Create the Week-5 baseline score
df["baseline_score"] = (
    df["gsc_impressions"].fillna(0) * 0.40 +
    df["gsc_clicks"].fillna(0) * 0.20 +
    df["ga4_sessions"].fillna(0) * 0.20 +
    df["ga4_engaged_sessions"].fillna(0) * 0.10 +
    df["gsc_avg_position"].fillna(999).apply(
        lambda x: 10 if 1 <= x <= 10 else (5 if 11 <= x <= 20 else 0)
    )
)

# Features used by W05
feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions"
]

X = df[feature_cols].copy()

X["gsc_avg_position"] = X["gsc_avg_position"].fillna(
    X["gsc_avg_position"].median()
)

X = X.fillna(0)

y = df["baseline_score"]

# Time-aware split
unique_dates = sorted(df["report_date"].unique())
split_index = int(len(unique_dates) * 0.80)
split_date = unique_dates[split_index]

train_mask = df["report_date"] < split_date
test_mask = df["report_date"] >= split_date

X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("Split date:", split_date)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

# Train Random Forest
model_w06 = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    max_depth=10
)

model_w06.fit(X_train, y_train)

predictions_w06 = model_w06.predict(X_test)

mae_w06 = mean_absolute_error(y_test, predictions_w06)

# Comparison
before_after = pd.DataFrame({
    "Validation": [
        "W05 model",
        "W06 audited model"
    ],
    "Split": [
        "Time-aware",
        "Time-aware"
    ],
    "Test rows": [
        len(X_test),
        len(X_test)
    ],
    "MAE": [
        0.34,
        round(mae_w06, 2)
    ]
})

display(Markdown("## Before / After Comparison"))
display(before_after)


# 2. My Model Under an Honest Split

## Validation design

The model is evaluated using a time-aware split. Earlier observations are
used for training and later observations are held out for testing.

This better reflects the real decision setting because the model should use
past information to make decisions about later observations.

The current experiment still uses the Week-4 baseline score as its measured
target. Therefore, this validation measures agreement with the baseline rule,
not future business performance.


Split date: 2026-06-25 00:00:00
Training rows: 3108522
Testing rows: 768370


## Before / After Comparison

,Validation,Split,Test rows,MAE
0,W05 model,Time-aware,768370,0.34
1,W06 audited model,Time-aware,768370,0.33


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
from IPython.display import Markdown, display

display(Markdown("""
# 3. Leakage Audit

The model should only use information that would have been available at the
time of the decision.

I therefore check the feature set for identifiers, future information,
labels, and derived outcome-related fields.
"""))

# Columns that should NOT be model features
excluded_columns = {
    "report_date": "Used only to create the time-aware split.",
    "client_hash_id": "Identifier; not a predictive content signal.",
    "content_hash_id": "Identifier; not a predictive content signal."
}

# Inspect the actual model feature set
audit_rows = []

for feature in feature_cols:
    if feature in excluded_columns:
        verdict = "EXCLUDE"
        reason = excluded_columns[feature]
    else:
        verdict = "KEEP"
        reason = "Observed performance signal available at prediction time."

    audit_rows.append({
        "Feature": feature,
        "Verdict": verdict,
        "Reason": reason
    })

leakage_audit = pd.DataFrame(audit_rows)

display(leakage_audit)

# Explicit checks
print("\nLeakage checks:")

print(
    "Identifier fields in model:",
    set(feature_cols) & {"report_date", "client_hash_id", "content_hash_id"}
)

future_terms = [
    "future", "target", "label", "outcome",
    "refresh", "updated", "trend"
]

future_like_features = [
    col for col in feature_cols
    if any(term in col.lower() for term in future_terms)
]

print("Future/label-like feature names:", future_like_features)

assert not (
    set(feature_cols)
    & {"report_date", "client_hash_id", "content_hash_id"}
)

assert len(future_like_features) == 0

display(Markdown("""
### Verdict: PASS

No identifier fields are used as model features, and the selected feature
names do not contain explicit future-window or label-derived fields.

`report_date` is used only for the honest time-aware split and is not supplied
to the model.

This audit supports the claim that the current feature set is designed to
avoid obvious leakage. It does not prove that every possible form of
information leakage has been eliminated.
"""))


# 3. Leakage Audit

The model should only use information that would have been available at the
time of the decision.

I therefore check the feature set for identifiers, future information,
labels, and derived outcome-related fields.


,Feature,Verdict,Reason
0,gsc_impressions,KEEP,Observed performance signal available at predi...
1,gsc_clicks,KEEP,Observed performance signal available at predi...
2,gsc_avg_position,KEEP,Observed performance signal available at predi...
3,ga4_sessions,KEEP,Observed performance signal available at predi...
4,ga4_engaged_sessions,KEEP,Observed performance signal available at predi...



Leakage checks:
Identifier fields in model: set()
Future/label-like feature names: []



### Verdict: PASS

No identifier fields are used as model features, and the selected feature
names do not contain explicit future-window or label-derived fields.

`report_date` is used only for the honest time-aware split and is not supplied
to the model.

This audit supports the claim that the current feature set is designed to
avoid obvious leakage. It does not prove that every possible form of
information leakage has been eliminated.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
from IPython.display import Markdown, display

display(Markdown("""
# 4. Claim Rewrite

## Original claim

> The Random Forest model improves content-refresh prioritization over the
> Week-4 baseline.

## Audit

This claim goes further than the evidence currently supports.

The W05 model achieved a low MAE against the Week-4 baseline, but that only
shows that the model can reproduce the baseline scoring pattern. We do not
yet have a verified future-window outcome showing that the model produces
better refresh decisions.

## Rewritten claim

> The Random Forest model **measured close agreement with the Week-4
> rule-based baseline under a time-aware evaluation split**.

The result is useful as a directional finding, but it should not be described
as proof that the model improves future content-refresh outcomes.

## Evidence boundary

What is supported:

- The model was evaluated using a time-aware split.
- The model's predictions were compared with the Week-4 baseline score.
- Feature leakage was explicitly audited.
- The model was heavily influenced by `gsc_impressions`.

What is not yet supported:

- That the model causes better content performance.
- That the model improves future refresh outcomes.
- That the model is superior to the baseline for business decisions.

A future-window outcome label and appropriate evaluation would be required
before making those stronger claims.
"""))


# 4. Claim Rewrite

## Original claim

> The Random Forest model improves content-refresh prioritization over the
> Week-4 baseline.

## Audit

This claim goes further than the evidence currently supports.

The W05 model achieved a low MAE against the Week-4 baseline, but that only
shows that the model can reproduce the baseline scoring pattern. We do not
yet have a verified future-window outcome showing that the model produces
better refresh decisions.

## Rewritten claim

> The Random Forest model **measured close agreement with the Week-4
> rule-based baseline under a time-aware evaluation split**.

The result is useful as a directional finding, but it should not be described
as proof that the model improves future content-refresh outcomes.

## Evidence boundary

What is supported:

- The model was evaluated using a time-aware split.
- The model's predictions were compared with the Week-4 baseline score.
- Feature leakage was explicitly audited.
- The model was heavily influenced by `gsc_impressions`.

What is not yet supported:

- That the model causes better content performance.
- That the model improves future refresh outcomes.
- That the model is superior to the baseline for business decisions.

A future-window outcome label and appropriate evaluation would be required
before making those stronger claims.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.